# Enriched MILP Ideas Ablation

Тест гипотез на уровне solver-ов (без внешних fallback/repair).

In [ ]:
from __future__ import annotations

from pathlib import Path
from datetime import datetime
import json
import sys
import pandas as pd

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'demo' else Path.cwd().resolve()
SRC_DIR = REPO_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from flowopt.solvers.enriched import (
    solve_enriched_milp_ablation_baseline,
    solve_enriched_milp_ablation_adaptive_k,
    solve_enriched_milp_ablation_penalty_sweep,
    solve_enriched_milp_ablation_zone_bundle,
    solve_enriched_milp_ablation_portfolio,
    solve_enriched_milp_lexicographic_2stage,
    solve_enriched_milp_critical_first,
    solve_enriched_milp_zone_quota,
    solve_enriched_milp_adaptive_hardness,
    solve_enriched_milp_time_windowed,
    solve_enriched_milp_lns_rounds,
    solve_enriched_milp_hybrid_seeded,
    solve_enriched_batched_greedy,
)


In [ ]:
DATASET_PATH = REPO_ROOT / 'demo' / 'data' / 'object_mass_feasible_fullfleet' / 'container_full_split_all_agents' / 'sweeps_task_agent_5pct' / 'dataset_real_spb_clean_full_split_by_containers_all_agents_with_distances_t001_a100.json'

if not DATASET_PATH.exists():
    raise FileNotFoundError(f'Not found: {DATASET_PATH}')

print('REPO_ROOT   :', REPO_ROOT)
print('DATASET_PATH:', DATASET_PATH)


In [ ]:
EXPERIMENTS = [
    {
        'exp_name': 'idea_baseline_strict_t30_k80',
        'fn': solve_enriched_milp_ablation_baseline,
        'kwargs': dict(time_limit_sec=30, max_pairs_per_task=80, unassigned_penalty=1e6),
    },
    {
        'exp_name': 'idea_lexicographic_2stage_t120',
        'fn': solve_enriched_milp_lexicographic_2stage,
        'kwargs': dict(time_budget_sec=120.0, max_pairs_per_task=180, unassigned_penalty=1e6),
    },
    {
        'exp_name': 'idea_adaptive_k_t90',
        'fn': solve_enriched_milp_ablation_adaptive_k,
        'kwargs': dict(time_budget_sec=90.0, pair_schedule=(40, 80, 120, 200), unassigned_penalty=1e6),
    },
    {
        'exp_name': 'idea_penalty_sweep_t90',
        'fn': solve_enriched_milp_ablation_penalty_sweep,
        'kwargs': dict(time_budget_sec=90.0, max_pairs_per_task=160, penalty_schedule=(1e5, 1e6, 1e7, 1e8)),
    },
    {
        'exp_name': 'idea_critical_first_t90',
        'fn': solve_enriched_milp_critical_first,
        'kwargs': dict(time_limit_sec=90, max_pairs_per_task=200, unassigned_penalty=1e6),
    },
    {
        'exp_name': 'idea_zone_quota_t90',
        'fn': solve_enriched_milp_zone_quota,
        'kwargs': dict(time_limit_sec=90, max_pairs_per_task=180, zone_min_coverage_ratio=0.92, unassigned_penalty=1e6),
    },
    {
        'exp_name': 'idea_adaptive_hardness_t90',
        'fn': solve_enriched_milp_adaptive_hardness,
        'kwargs': dict(time_limit_sec=90, max_pairs_per_task=200, unassigned_penalty=1e6),
    },
    {
        'exp_name': 'idea_time_windowed_t120',
        'fn': solve_enriched_milp_time_windowed,
        'kwargs': dict(time_budget_sec=120.0, max_pairs_per_task=180, unassigned_penalty=1e6),
    },
    {
        'exp_name': 'idea_lns_rounds_t120',
        'fn': solve_enriched_milp_lns_rounds,
        'kwargs': dict(time_budget_sec=120.0, max_pairs_per_task=180, unassigned_penalty=1e6, max_rounds=6),
    },
    {
        'exp_name': 'idea_hybrid_seeded_t120',
        'fn': solve_enriched_milp_hybrid_seeded,
        'kwargs': dict(time_budget_sec=120.0, max_pairs_per_task=200, unassigned_penalty=1e6),
    },
    {
        'exp_name': 'idea_zone_bundle_tasks',
        'fn': solve_enriched_milp_ablation_zone_bundle,
        'kwargs': dict(time_limit_sec_per_zone=20, max_pairs_per_bundle=120, bundle_fill_factor=0.90, bundle_max_tasks=8, objective='tasks'),
    },
    {
        'exp_name': 'idea_portfolio_t60',
        'fn': solve_enriched_milp_ablation_portfolio,
        'kwargs': dict(time_budget_sec=60.0, max_starts=12, per_start_time_limit_sec=8, max_pairs_per_task=80, jitter=0.08, seed=42),
    },
    {
        'exp_name': 'reference_batched_greedy',
        'fn': solve_enriched_batched_greedy,
        'kwargs': dict(top_k_agents=30, balance_penalty=0.02, random_seed=42),
    },
]


In [ ]:
results = []

for cfg in EXPERIMENTS:
    exp_name = cfg['exp_name']
    fn = cfg['fn']
    kwargs = cfg['kwargs']
    print(f"[RUN] {exp_name}")
    res = fn(dataset_path=DATASET_PATH, **kwargs)
    d = res.as_dict()
    checks = (d.get('details') or {}).get('checks') or {}

    row = {
        'exp_name': exp_name,
        'algorithm': d.get('algorithm'),
        'feasible': d.get('feasible'),
        'assigned_tasks': d.get('assigned_routes'),
        'assigned_trips': d.get('assigned_trips'),
        'unassigned_tasks': d.get('unassigned_tasks'),
        'active_agents': d.get('active_agents'),
        'transport_work_ton_km': d.get('transport_work_ton_km'),
        'total_km': d.get('total_km'),
        'deadhead_share_pct': d.get('deadhead_share_pct'),
        'total_hours': d.get('total_hours'),
        'runtime_sec': d.get('runtime_sec'),
        'all_checks_ok': checks.get('all_checks_ok'),
        'daily_limits_ok': checks.get('daily_limits_ok'),
        'object_limits_ok': checks.get('object_limits_ok'),
        'compatibility_ok': checks.get('compatibility_ok'),
        'reachability_ok': checks.get('reachability_ok'),
        'overflow_km_agents': checks.get('overflow_km_agents'),
        'overflow_hours_agents': checks.get('overflow_hours_agents'),
        'object_mass_violations': checks.get('object_mass_violations'),
        'object_volume_violations': checks.get('object_volume_violations'),
        'solver_error': d.get('solver_error'),
    }
    results.append(row)
    print(f"[DONE] {exp_name}: feasible={row['feasible']} unassigned={row['unassigned_tasks']} runtime={row['runtime_sec']}s")

summary = pd.DataFrame(results)
summary = summary.sort_values(
    by=['all_checks_ok', 'unassigned_tasks', 'runtime_sec', 'total_km'],
    ascending=[False, True, True, True],
).reset_index(drop=True)
summary


In [ ]:
TIME_ABLATION = []

for budget in [20, 40, 80]:
    for name, fn, kwargs in [
        ('baseline', solve_enriched_milp_ablation_baseline, dict(time_limit_sec=budget, max_pairs_per_task=120)),
        ('lexicographic_2stage', solve_enriched_milp_lexicographic_2stage, dict(time_budget_sec=float(budget), max_pairs_per_task=160)),
        ('adaptive_k', solve_enriched_milp_ablation_adaptive_k, dict(time_budget_sec=float(budget), pair_schedule=(40, 80, 120, 200))),
        ('critical_first', solve_enriched_milp_critical_first, dict(time_limit_sec=budget, max_pairs_per_task=180)),
        ('adaptive_hardness', solve_enriched_milp_adaptive_hardness, dict(time_limit_sec=budget, max_pairs_per_task=180)),
        ('portfolio', solve_enriched_milp_ablation_portfolio, dict(time_budget_sec=float(budget), max_starts=8, per_start_time_limit_sec=max(5, budget // 8), max_pairs_per_task=100, seed=42)),
        ('zone_bundle', solve_enriched_milp_ablation_zone_bundle, dict(time_limit_sec_per_zone=max(4, budget // 4), max_pairs_per_bundle=120, bundle_fill_factor=0.9, bundle_max_tasks=8)),
    ]:
        res = fn(dataset_path=DATASET_PATH, **kwargs)
        d = res.as_dict()
        checks = (d.get('details') or {}).get('checks') or {}
        TIME_ABLATION.append({
            'method': name,
            'time_budget_sec': budget,
            'algorithm': d.get('algorithm'),
            'feasible': d.get('feasible'),
            'unassigned_tasks': d.get('unassigned_tasks'),
            'assigned_tasks': d.get('assigned_routes'),
            'runtime_sec': d.get('runtime_sec'),
            'total_km': d.get('total_km'),
            'all_checks_ok': checks.get('all_checks_ok'),
        })

abl_df = pd.DataFrame(TIME_ABLATION)
abl_df.sort_values(['method', 'time_budget_sec'])


In [ ]:
pivot_cov = abl_df.pivot_table(index='time_budget_sec', columns='method', values='assigned_tasks', aggfunc='max')
pivot_unassigned = abl_df.pivot_table(index='time_budget_sec', columns='method', values='unassigned_tasks', aggfunc='min')

print('Assigned tasks by time budget:')
display(pivot_cov)
print('Unassigned tasks by time budget:')
display(pivot_unassigned)


In [ ]:
OUT_DIR = REPO_ROOT / 'demo' / 'local' / 'enriched_milp_ideas_ablation'
OUT_DIR.mkdir(parents=True, exist_ok=True)
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
out_path = OUT_DIR / f'ablation_{ts}.json'

payload = {
    'dataset_path': str(DATASET_PATH),
    'created_at': ts,
    'summary': results,
    'time_ablation': TIME_ABLATION,
}
out_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
print('Saved:', out_path)
